# A1 — Regresijsko drevo, naključni gozd in GBT: Olist

Napovedujemo **čas od naročila do dostave v dnevih**. Ena vrstica je naročilo.
Začnemo z razložljivim majhnim drevesom, nato uporabimo naključni gozd in gradientno pospeševanje.
Postopka obeh ansamblov že poznamo iz klasifikacije; tukaj pogledamo spremembe pri regresiji.

Celice izvajajte po vrsti. Pri oznaki **PREIZKUSITE** lahko spreminjate nastavitve.
Rezultati se zbirajo v skupni tabeli. Skupna funkcija po potrebi pripravi podatke. Testne tabele ne odpiramo.

Podatke po potrebi pripravi `poskrbi_za_modeliranje`; `preberi_tabelo` jih bere s tipi iz sheme.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)

In [ ]:
import json
from pathlib import Path
from time import perf_counter

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, PolynomialFeatures,
                                   StandardScaler)
from sklearn.tree import DecisionTreeRegressor, plot_tree

from utils.modeliranje import (poskrbi_za_modeliranje, preberi_shemo,
                               preberi_tabelo)

PODATKI = poskrbi_za_modeliranje("olist")
SEME, N_JOBS = 42, 2
pd.set_option("display.max_rows", 14)
pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10,4), "font.size": 11})
print("LightGBM:", lgb.__version__)

## 1. Ohranimo časovno delitev
Učimo se na naročilih pred marcem 2018, katerih dostava je bila takrat že znana.
Validacija vsebuje naročila od marca do maja, dostavljena pred junijem.
Datum dostave uporabimo samo za preverjanje pravil delitve in cilja; **ni vhod v model**.

Napoved je mišljena ob naročilu. ID-ji, končni status, ocene kupcev in podatki o dejanski dostavi niso značilke.
Podatki zajemajo naročila z znano uspešno dostavo, ne vseh možnih naročil. Zahteva po znanem izidu ob meji lahko
izloči dolge dostave blizu meje; rezultat zato opisuje ta pripravljeni vzorec.

In [ ]:
shema = preberi_shemo(PODATKI / "shema.json")
CILJ, KLJUC = shema["cilj"], shema["kljuc"]
DATUMI = shema["datumski_stolpci"]
train = preberi_tabelo(PODATKI / "train.csv", shema)
val = preberi_tabelo(PODATKI / "val.csv", shema)
VSE = shema["znacilke"]
KAT = shema["kategoricne_znacilke"]
assert train[KLJUC].is_unique and val[KLJUC].is_unique
assert set(train[KLJUC]).isdisjoint(val[KLJUC])
assert not set(shema["metapodatki"]+[CILJ]) & set(VSE)
meja_val = pd.Timestamp(shema["delitev"]["zacetek_val"])
meja_test = pd.Timestamp(shema["delitev"]["zacetek_test"])
assert (train["datum_nakupa"] < meja_val).all()
assert (train["datum_znanega_izida"] < meja_val).all()
assert (val["datum_nakupa"] >= meja_val).all() and (val["datum_nakupa"] < meja_test).all()
assert (val["datum_znanega_izida"] < meja_test).all()
for df in [train,val]:
    assert df[CILJ].notna().all() and np.isfinite(df[CILJ]).all() and (df[CILJ]>=0).all()
    izracunani_dnevi = (df["datum_znanega_izida"]-df["datum_nakupa"]).dt.total_seconds()/86400
    np.testing.assert_allclose(df[CILJ],izracunani_dnevi,atol=1e-9)
X_train, X_val = train[VSE].copy(), val[VSE].copy()
y_train, y_val = train[CILJ].copy(), val[CILJ].copy()
for X in [X_train,X_val]:
    assert not X.isna().any().any(), "Uporabite dopolnjene tabele iz prejšnjega poglavja."
    X[KAT] = X[KAT].astype(str)
    assert np.isfinite(X[[s for s in VSE if s not in KAT]].to_numpy(float)).all()
display(pd.DataFrame([
    {"zbirka":ime,"naročil":len(df),"prvo naročilo":df.datum_nakupa.min(),
     "zadnje naročilo":df.datum_nakupa.max(),"mediana dni":df[CILJ].median(),"povprečje dni":df[CILJ].mean()}
    for ime,df in [("učna",train),("validacijska",val)]
]).round(2))

### Priprava značilk in pravila primerjave
Za začetno primerjavo izberemo nabor **košarica in lokacija**: velikost in vrednost naročila,
poštnina, teža ter lokaciji kupca in prodajalca. Isti nabor dobijo vsi začetni modeli.
Za samostojno vajo je na voljo celoten pripravljeni nabor.

Številke so že dopolnjene z učnimi medianami. Za drevesa jih ne standardiziramo.
Kategorije kodiramo z one-hot kodiranjem, naučenim samo na učnih podatkih.
Za linearno in polinomsko primerjavo pozneje uporabimo ustrezno skaliranje znotraj njunih postopkov.

In [ ]:
OSNOVNE = list(shema["nabori_znacilk"]["kosarica_in_lokacija"])

def pripravi(nabor):
    num, kat = [s for s in nabor if s not in KAT], [s for s in nabor if s in KAT]
    prep = ColumnTransformer([
        ("num","passthrough",num),
        ("kat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),kat),
    ],remainder="drop")
    xt = prep.fit_transform(X_train[nabor])
    xv = prep.transform(X_val[nabor])
    imena = prep.get_feature_names_out()
    izvor = list(num)
    if kat:
        for s,kategorije in zip(kat,prep.named_transformers_["kat"].categories_):
            izvor.extend([s]*len(kategorije))
    assert len(imena)==len(izvor)==xt.shape[1]==xv.shape[1]
    return prep,xt,xv,imena,np.array(izvor)

priprava, Xt, Xv, IMENA, IZVOR = pripravi(OSNOVNE)
print("Začetne značilke:",OSNOVNE)
print("Obliki po kodiranju:",Xt.shape,Xv.shape)

### Izhodišče: vsem napovemo učno mediano
**MAE** pove povprečno absolutno napako v dnevih. **RMSE** je prav tako v dnevih, vendar bolj poudari velike napake.
Za orientacijo dodamo R²; lahko je tudi negativen.

Konstantna napoved, ki minimizira učno absolutno napako, je mediana. Pri kvadratni izgubi je to povprečje.
Za glavne primerjave uporabimo MAE; modelov s kvadratno izgubo zato ne enačimo z neposrednim optimiziranjem MAE.

In [ ]:
POSKUSI = {}  # Ponovna izvedba te celice počisti dosedanjo primerjavo.

def metrike(y,p):
    p=np.asarray(p,dtype=float)
    assert p.shape==np.asarray(y).shape and np.isfinite(p).all()
    return {"MAE":mean_absolute_error(y,p),"RMSE":np.sqrt(mean_squared_error(y,p)),"R²":r2_score(y,p)}

def shrani(ime,model,prep,nabor,pt,pv,sekunde,**dodatno):
    ucne,validacijske=metrike(y_train,pt),metrike(y_val,pv)
    rezultat=dict(ime=ime,model=model,priprava=prep,nabor=list(nabor),p_train=np.asarray(pt),p_val=np.asarray(pv),
        povzetek={"model":ime,"značilk":len(nabor),"učni MAE":ucne["MAE"],"učni RMSE":ucne["RMSE"],
                  "validacijski MAE":validacijske["MAE"],"validacijski RMSE":validacijske["RMSE"],
                  "R²":validacijske["R²"],"negativnih napovedi":int((np.asarray(pv)<0).sum()),"učenje (s)":sekunde},**dodatno)
    POSKUSI[ime]=rezultat
    return rezultat

def primerjaj(imena=None):
    izbrani=POSKUSI.values() if imena is None else [POSKUSI[i] for i in imena]
    tabela=pd.DataFrame([p["povzetek"] for p in izbrani]).set_index("model")
    display(tabela.sort_values("validacijski MAE").round(4))
    return tabela

izh=DummyRegressor(strategy="median").fit(Xt,y_train)
shrani("Učna mediana",izh,priprava,OSNOVNE,izh.predict(Xt),izh.predict(Xv),0)
print(f"Vsem napovemo {float(izh.constant_.item()):.2f} dni.")
rezultati=primerjaj()

## 2. Prvo regresijsko drevo
Za izris uporabimo največjo globino 2 in vsaj 100 učnih naročil na list.
Kriterij `squared_error` izbira razcepe, ki zmanjšujejo kvadratno napako.
**Vrednost v listu je povprečje ciljnih vrednosti učnih naročil v tem listu.**

In [ ]:
start=perf_counter()
drevo=DecisionTreeRegressor(max_depth=2,min_samples_leaf=100,criterion="squared_error",random_state=SEME)
drevo.fit(Xt,y_train)
cas=perf_counter()-start
plitvo=shrani("Plitvo drevo",drevo,priprava,OSNOVNE,drevo.predict(Xt),drevo.predict(Xv),cas,imena=IMENA,izvor=IZVOR)
fig,ax=plt.subplots(figsize=(15,6))
plot_tree(drevo,feature_names=IMENA,filled=True,rounded=True,precision=3,fontsize=9,ax=ax)
ax.set_title("Regresijsko drevo · value je napoved v dnevih")
plt.tight_layout()
plt.show()
rezultati=primerjaj()

**Kako beremo prikaz:** leva veja pomeni, da pogoj ≤ velja. `samples` je število učnih naročil,
`value` njihovo povprečje ciljnih vrednosti, `squared_error` pa povprečna kvadratna razpršenost okoli tega povprečja.
Pri one-hot značilki razcep ločuje prisotnost in odsotnost kategorije. Oznake v grafu so zaokrožene.

## 3. Razložimo napoved konkretnega naročila
**PREIZKUSITE:** spremenite `VRSTICA` in ponovite naslednje tri celice.
Naročilo izberemo po položaju, ne po velikosti njegove napake. V listu pregledujemo samo **učne** izide.

In [ ]:
VRSTICA=0
assert 0<=VRSTICA<len(Xv)
# Scikit-learn pri prehodu skozi drevo uporablja float32.
vrstica=Xv[VRSTICA].astype(np.float32)
t=drevo.tree_
vozlisce=0
pot=[]
while t.children_left[vozlisce]!=t.children_right[vozlisce]:
    j=int(t.feature[vozlisce]); prag=float(t.threshold[vozlisce]); vrednost=float(vrstica[j])
    levo=vrednost<=prag
    pot.append({"vozlišče":vozlisce,"značilka":IMENA[j],"vrednost naročila":vrednost,"prag ≤":prag,
                "veja":"levo" if levo else "desno"})
    vozlisce=int(t.children_left[vozlisce] if levo else t.children_right[vozlisce])
list_id=vozlisce
assert list_id==int(drevo.apply(Xv[[VRSTICA]])[0])
ucni_listi=drevo.apply(Xt)
v_listu=ucni_listi==list_id
izidi_lista=y_train.loc[v_listu]
napoved=float(drevo.predict(Xv[[VRSTICA]])[0])
np.testing.assert_allclose(napoved,izidi_lista.mean(),atol=1e-10)
assert len(izidi_lista)==t.n_node_samples[list_id]
display(pd.DataFrame(pot).round(4))
print(f"List {list_id}: {len(izidi_lista)} učnih naročil, povprečje = napoved = {napoved:.3f} dni.")
print(f"Dejanska dostava izbranega validacijskega naročila: {y_val.iloc[VRSTICA]:.3f} dni.")

In [ ]:
display(izidi_lista.describe(percentiles=[.1,.25,.5,.75,.9]).to_frame("dni dostave").round(3))
# Nekaj dejanskih učnih primerov iz doseženega lista; noben validacijski izid ni v povprečju.
display(train.loc[v_listu,["datum_nakupa","vrednost","postnina","zvezna_drzava",CILJ]].head(6))
fig,ax=plt.subplots()
ax.hist(izidi_lista,bins=40,color="#2993ad",alpha=.75)
ax.axvline(napoved,color="#b64b44",lw=2,label=f"Povprečje / napoved {napoved:.1f} dni")
ax.axvline(izidi_lista.median(),color="#94ad35",lw=2,label=f"Mediana {izidi_lista.median():.1f} dni")
ax.set(xlabel="Dejanski čas dostave v učnem listu (dni)",ylabel="Število naročil",title="Enaka napoved za različne izide")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Povprečje minimizira kvadratno napako; mediana absolutno napako.
a=izidi_lista.to_numpy()
primerjava_lista=pd.DataFrame([
    {"konstanta":ime,"napoved":v,"MAE v listu":np.mean(np.abs(a-v)),"MSE v listu":np.mean((a-v)**2)}
    for ime,v in [("Povprečje",a.mean()),("Mediana",np.median(a))]
])
display(primerjava_lista.round(4))

Povprečje v listu ni zagotovljeni čas dostave. Razpršenost opisuje učne primere v skupini;
prikazani percentili niso samodejno preverjen napovedni interval za novo naročilo.
Pot razloži izračun modela, ne vzročnega vpliva poštnine ali drugih značilk na dostavo.

## 4. Kako globoko naj raste drevo?
Naučimo več dreves na istih vrsticah in značilkah. V prvem prikazu spreminjamo samo globino.
Podpora lista ostane 30. Vse napake primerjamo na isti časovni validaciji.

In [ ]:
GLOBINE=[2,4,6,10,None]  # None pomeni brez omejitve globine; velikost lista še vedno omejuje rast.
globine=[]
for globina in GLOBINE:
    start=perf_counter()
    m=DecisionTreeRegressor(max_depth=globina,min_samples_leaf=30,criterion="squared_error",random_state=SEME).fit(Xt,y_train)
    cas=perf_counter()-start
    p=shrani(f"Drevo globina {globina}",m,priprava,OSNOVNE,m.predict(Xt),m.predict(Xv),cas,imena=IMENA,izvor=IZVOR)
    globine.append({"omejitev":"brez" if globina is None else str(globina),"dejanska globina":m.get_depth(),"listov":m.get_n_leaves(),
                    "učni MAE":p["povzetek"]["učni MAE"],"validacijski MAE":p["povzetek"]["validacijski MAE"]})
tabela_globin=pd.DataFrame(globine)
display(tabela_globin.round(3))
fig,ax=plt.subplots()
for metrika in ["učni MAE","validacijski MAE"]:
    ax.plot(tabela_globin["omejitev"],tabela_globin[metrika],marker="o",label=metrika)
ax.set(xlabel="Največja globina",ylabel="MAE (dni)",title="Večje drevo in posploševanje")
ax.legend();plt.tight_layout();plt.show()

**Vprašanja:** Ali večje drevo zmanjša tudi validacijsko napako? Zakaj je lahko učna napaka precej manjša?
Učenje minimizira kvadratno napako, zato ni nujno, da učni MAE z globino vedno pada.

In [ ]:
# PREIZKUSITE: manjše / večje liste pri isti globini.
VELIKOSTI_LISTA=[5,30,200]
podpora=[]
for najmanj in VELIKOSTI_LISTA:
    m=DecisionTreeRegressor(max_depth=8,min_samples_leaf=najmanj,random_state=SEME).fit(Xt,y_train)
    podpora.append({"min_samples_leaf":najmanj,"listov":m.get_n_leaves(),
                    "učni MAE":mean_absolute_error(y_train,m.predict(Xt)),
                    "validacijski MAE":mean_absolute_error(y_val,m.predict(Xv))})
display(pd.DataFrame(podpora).round(3))

### Kratek prikaz na umetnih podatkih: stopnice in ekstrapolacija
Ta ločeni primer ima samo eno značilko, zato lahko narišemo celotno napovedno funkcijo.
Zunaj učnega območja drevo doseže enega od obstoječih listov. Ne nadaljuje trenda kot premica.
Enako velja za standardni naključni gozd in GBT s konstantnimi vrednostmi v listih, če ostali vhodi ostanejo fiksni.

In [ ]:
rng=np.random.default_rng(SEME)
x_um=np.linspace(1,10,100).reshape(-1,1)
y_um=3+1.4*x_um[:,0]+rng.normal(0,1.2,len(x_um))
m_um=DecisionTreeRegressor(max_depth=3,min_samples_leaf=8,random_state=SEME).fit(x_um,y_um)
lin_um=LinearRegression().fit(x_um,y_um)
mreza=np.linspace(-1,14,300).reshape(-1,1)
fig,ax=plt.subplots()
ax.scatter(x_um[:,0],y_um,s=15,alpha=.6,label="Učni podatki")
ax.plot(mreza[:,0],m_um.predict(mreza),label="Regresijsko drevo",lw=2)
ax.plot(mreza[:,0],lin_um.predict(mreza),label="Premica",lw=2)
ax.axvspan(-1,1,color="grey",alpha=.1);ax.axvspan(10,14,color="grey",alpha=.1)
ax.set(xlabel="Umetna značilka",ylabel="Umetni cilj",title="Umetni primer · sivo je zunaj učnega območja")
ax.legend();plt.tight_layout();plt.show()

## 5. Naključni gozd: povprečje napovedi dreves
Uporabimo `RandomForestRegressor`. Drevesa učimo na vzorčenih učnih podatkih in z izbiro značilk pri razcepih.
Vsako drevo napove **število dni**, gozd pa napovedi povpreči. OOB ocene ne uporabljamo kot nadomestila za časovno validacijo.

In [ ]:
RF_PARAMETRI=dict(n_estimators=80,max_depth=12,min_samples_leaf=30,max_features=.8)
start=perf_counter()
gozd=RandomForestRegressor(**RF_PARAMETRI,criterion="squared_error",bootstrap=True,
                           random_state=SEME,n_jobs=N_JOBS).fit(Xt,y_train)
cas=perf_counter()-start
rf=shrani("Naključni gozd",gozd,priprava,OSNOVNE,gozd.predict(Xt),gozd.predict(Xv),cas,imena=IMENA,izvor=IZVOR)
rezultati=primerjaj(["Učna mediana","Plitvo drevo","Naključni gozd"])

In [ ]:
# Ista VRSTICA kot pri razlagi drevesa; po želji jo spremenite.
napovedi_dreves=np.array([m.predict(Xv[[VRSTICA]])[0] for m in gozd.estimators_])
napoved_gozda=float(gozd.predict(Xv[[VRSTICA]])[0])
np.testing.assert_allclose(napoved_gozda,napovedi_dreves.mean(),atol=1e-10)
print(f"Povprečje {len(napovedi_dreves)} dreves = napoved gozda = {napoved_gozda:.3f} dni.")
fig,ax=plt.subplots()
ax.hist(napovedi_dreves,bins=15,color="#2993ad",alpha=.8)
ax.axvline(napoved_gozda,color="#b64b44",lw=2,label="Napoved gozda")
ax.set(xlabel="Napoved posameznega drevesa (dni)",ylabel="Število dreves",title="Različna drevesa za isto naročilo")
ax.legend();plt.tight_layout();plt.show()

Razpršenost napovedi dreves **ni samodejno interval zaupanja ali napovedni interval**.
Drevesa so povezana, njihova razpršenost pa ne zajame vseh virov napake. Tukaj prikaz služi razumevanju povprečenja.

## 6. GBT: začetno povprečje in popravki v dnevih
Uporabimo `LGBMRegressor(objective="regression")`, torej kvadratno izgubo.
Začetna ocena je učno povprečje. Prispevki dreves popravljajo napoved neposredno v dnevih; **sigmoida ni**.

Kot glavno validacijsko merilo izberemo **MAE**. Zgodnje ustavljanje spremlja samo to merilo;
za prikaz shranimo tudi RMSE. Učna izguba in metrika za izbor modela zato nista isti.

In [ ]:
GBT_PARAMETRI=dict(n_estimators=800,learning_rate=.05,num_leaves=15,max_depth=5,
                   min_child_samples=100,reg_lambda=1.0)
POTRPEZLJIVOST=40

def nauci_gbt(xt,xv,nastavitve):
    m=lgb.LGBMRegressor(objective="regression",metric="None",**nastavitve,
        boost_from_average=True,random_state=SEME,n_jobs=N_JOBS,
        deterministic=True,force_col_wise=True,verbosity=-1)
    m.fit(xt,y_train,eval_X=(xt,xv),eval_y=(y_train,y_val),
          eval_names=["ucna","validacijska"],eval_metric=["l1","rmse"],
          callbacks=[lgb.log_evaluation(0),lgb.early_stopping(POTRPEZLJIVOST,first_metric_only=True,verbose=False)])
    k=m.best_iteration_
    assert k==int(np.argmin(m.evals_result_["validacijska"]["l1"]))+1
    np.testing.assert_allclose(mean_absolute_error(y_val,m.predict(xv,num_iteration=k)),
                               m.evals_result_["validacijska"]["l1"][k-1],atol=1e-8)
    return m,k

start=perf_counter()
gbt,K=nauci_gbt(Xt,Xv,GBT_PARAMETRI)
cas=perf_counter()-start
gbt_rez=shrani("GBT",gbt,priprava,OSNOVNE,gbt.predict(Xt,num_iteration=K),gbt.predict(Xv,num_iteration=K),
               cas,iteracija=K,imena=IMENA,izvor=IZVOR)
print("Izbrana iteracija:",K,"; izvedenih korakov:",len(gbt.evals_result_["validacijska"]["l1"]))
if len(gbt.evals_result_["validacijska"]["l1"])>=GBT_PARAMETRI["n_estimators"]:
    print("Dosežen je proračun dreves; daljše učenje bi lahko še pomagalo.")
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,metrika,naslov in zip(axes,["l1","rmse"],["MAE","RMSE"]):
    for kljuc,oznaka in [("ucna","Učna"),("validacijska","Validacijska")]:
        ys=gbt.evals_result_[kljuc][metrika]
        ax.plot(np.arange(1,len(ys)+1),ys,label=oznaka)
    ax.axvline(K,color="#168a87",ls="--",label=f"Izbor po MAE: {K}")
    ax.set(xlabel="Število dreves",ylabel=f"{naslov} (dni)",title=naslov)
    ax.legend()
plt.tight_layout();plt.show()

### Preverimo napoved GBT iz doseženih listov
Kot v klasifikacijskem prikazu LightGBM začetno povprečje shrani v liste prvega drevesa;
listne vrednosti že vključujejo hitrost učenja. Zato ne prištejemo povprečja ali množimo s hitrostjo učenja še enkrat.
Za didaktični prikaz začetno povprečje ločimo od prvega popravka.

In [ ]:
listi_gbt=np.asarray(gbt.predict(Xv[[VRSTICA]],pred_leaf=True,num_iteration=K)).reshape(1,-1)[0]
listne_vrednosti=np.array([gbt.booster_.get_leaf_output(i,int(l)) for i,l in enumerate(listi_gbt)])
zacetno_povprecje=float(y_train.mean())
popravki=listne_vrednosti.copy();popravki[0]-=zacetno_povprecje
vmesne=zacetno_povprecje+np.cumsum(popravki)
napoved_gbt=float(gbt.predict(Xv[[VRSTICA]],num_iteration=K)[0])
np.testing.assert_allclose(listne_vrednosti.sum(),napoved_gbt,atol=1e-8)
np.testing.assert_allclose(vmesne[-1],napoved_gbt,atol=1e-8)
for k in sorted(set([1,min(3,K),K])):
    np.testing.assert_allclose(vmesne[k-1],gbt.predict(Xv[[VRSTICA]],num_iteration=k)[0],atol=1e-8)
display(pd.DataFrame({"drevo":np.arange(1,K+1),"list":listi_gbt,"shranjena vrednost":listne_vrednosti,
                      "popravek":popravki,"skupaj dni":vmesne}).head(6).round(4))
print(f"Začetno povprečje {zacetno_povprecje:.3f} + vsota popravkov {popravki.sum():+.3f} = {napoved_gbt:.3f} dni.")
print("Preverjeno: vsota in napoved modela se ujemata; ni pretvorbe s sigmoidom.")

## 7. Primerjava vseh modelov na istih naročilih
Za povezavo s prejšnjim delom izračunamo še linearno in polinomsko regresijo na **istem začetnem naboru in isti delitvi**.
To je samostojna rezervna primerjava: ne prevzemamo številk iz drugih notebookov.
Polinom stopnje 2 oblikujemo samo iz numeričnih značilk; kategorije ostanejo one-hot.

Izvedba je pripravljena, razlage teh dveh metod ne ponavljamo. Če že imate primerljivo izvedbo iz prejšnjega sklopa,
lahko ta del preskočite. Ne mešajte rezultatov z drugih delitev ali drugače definiranega cilja.

In [ ]:
# PREIZKUSITE po želji; pri stopnji 2 uporabimo isti kompaktni nabor kot ostali začetni modeli.
IZVEDI_LINEARNO_PRIMERJAVO=True
if IZVEDI_LINEARNO_PRIMERJAVO:
    num=[s for s in OSNOVNE if s not in KAT];kat=[s for s in OSNOVNE if s in KAT]
    for stopnja,ime in [(1,"Linearna regresija"),(2,"Polinomska regresija (2)")]:
        numericni=Pipeline([("scale",StandardScaler()),
                            ("poly",PolynomialFeatures(stopnja,include_bias=False)),
                            ("scale_poly",StandardScaler())])
        prep=ColumnTransformer([("num",numericni,num),
            ("kat",OneHotEncoder(handle_unknown="ignore",drop="first",sparse_output=False),kat)])
        postopek=Pipeline([("priprava",prep),("model",LinearRegression())])
        start=perf_counter();postopek.fit(X_train[OSNOVNE],y_train);cas=perf_counter()-start
        shrani(ime,postopek,None,OSNOVNE,postopek.predict(X_train[OSNOVNE]),postopek.predict(X_val[OSNOVNE]),cas)
GLAVNI=["Učna mediana","Plitvo drevo","Naključni gozd","GBT"]
GLAVNI += [ime for ime in ["Linearna regresija","Polinomska regresija (2)"] if ime in POSKUSI]
rezultati=primerjaj(GLAVNI)

Napovedi pustimo nespremenjene, tudi če bi kateri model napovedal negativen čas.
To omejitev pokažemo v tabeli; morebitno omejitev na nič bi morali obravnavati in preveriti kot del postopka.
**Najboljši MAE ne zagotavlja najboljšega RMSE.** Velike napake imajo pri RMSE večjo težo.
Časi učenja so okvirni in odvisni od računalnika; pri linearnih postopkih vključujejo tudi njuno pripravo značilk.

In [ ]:
# Isti prikazni vzorec za vse modele; metrike zgoraj uporabljajo vseh 18.117 validacijskih naročil.
prikaz_i=np.random.default_rng(SEME).choice(len(y_val),size=min(2500,len(y_val)),replace=False)
fig,axes=plt.subplots(1,3,figsize=(14,4.2))
for ax,ime in zip(axes,["Plitvo drevo","Naključni gozd","GBT"]):
    p=POSKUSI[ime]["p_val"][prikaz_i];y=y_val.iloc[prikaz_i].to_numpy()
    ax.scatter(y,p,s=8,alpha=.25)
    spodaj=min(0,float(p.min()));zgoraj=max(float(y.max()),float(p.max()))
    ax.plot([spodaj,zgoraj],[spodaj,zgoraj],ls="--",color="grey")
    ax.set(title=ime,xlabel="Dejansko (dni)",ylabel="Napoved (dni)",xlim=(spodaj,zgoraj),ylim=(spodaj,zgoraj))
plt.tight_layout();plt.show()

### Kje se model najbolj zmoti?
**PREIZKUSITE:** zamenjajte `MODEL_ZA_NAPAKE`. Predznak definiramo kot **napoved − dejanski čas**:
negativna napaka pomeni, da smo dostavo napovedali prehitro.
Datum dejanske dostave ostane samo pri analizi, nikoli med vhodi.

In [ ]:
MODEL_ZA_NAPAKE="GBT"
p=POSKUSI[MODEL_ZA_NAPAKE]["p_val"]
napake=val[[KLJUC,"datum_nakupa","zvezna_drzava","vrednost","postnina",CILJ]].copy()
napake["napoved_dni"]=p
napake["napaka"]=p-y_val.to_numpy()
napake["absolutna_napaka"]=napake.napaka.abs()
display(napake.nlargest(5,"absolutna_napaka").round(3))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(napake.napaka,bins=60,color="#2993ad")
axes[0].axvline(0,color="grey",ls="--")
axes[0].set(xlabel="Napoved − dejanski čas (dni)",ylabel="Število naročil",title="Porazdelitev napak")
axes[1].scatter(napake[CILJ].iloc[prikaz_i],napake.napaka.iloc[prikaz_i],s=8,alpha=.25)
axes[1].axhline(0,color="grey",ls="--")
axes[1].set(xlabel="Dejansko trajanje (dni)",ylabel="Napaka (dni)",title="Napake pri kratkih in dolgih dostavah")
plt.tight_layout();plt.show()
# Pregled po mesecih: majhnih skupin ne obravnavamo kot zanesljivih ocen.
po_mesecih=napake.assign(mesec=napake.datum_nakupa.dt.to_period("M").astype(str)).groupby("mesec").agg(
    narocil=("napaka","size"),MAE=("absolutna_napaka","mean"),povprecna_napaka=("napaka","mean"))
display(po_mesecih.round(3))

### Katere značilke model uporablja?
Pri drevesu in gozdu pokažemo normalizirano zmanjšanje nečistosti (**MDI**); pri kvadratni izgubi temelji na zmanjšanju variance.
Pri GBT uporabimo **split count** in **skupni gain**. Teh številk ne primerjamo neposredno med različnimi definicijami.
One-hot stolpce seštejemo pod prvotno značilko.

In [ ]:
mdi=pd.DataFrame({"značilka":IZVOR,"Drevo MDI":drevo.feature_importances_,"Gozd MDI":gozd.feature_importances_})
mdi=mdi.groupby("značilka").sum()
gbt_pom=pd.DataFrame({"značilka":IZVOR,
    "split count":gbt.booster_.feature_importance(importance_type="split",iteration=K),
    "skupni gain":gbt.booster_.feature_importance(importance_type="gain",iteration=K)}).groupby("značilka").sum()
fig,axes=plt.subplots(1,3,figsize=(15,4.8))
for ax,(tabela,stolpec) in zip(axes,[(mdi,"Gozd MDI"),(gbt_pom,"split count"),(gbt_pom,"skupni gain")]):
    s=tabela[stolpec].nlargest(8).sort_values()
    ax.barh(s.index,s,color="#2993ad")
    ax.set(title=stolpec,xlabel=stolpec)
plt.tight_layout();plt.show()
display(mdi.sort_values("Gozd MDI",ascending=False).head(8).round(4))

Pomembnost opisuje uporabo značilk v naučenem modelu, ne smeri vpliva ali vzročnosti.
Povezane značilke se lahko nadomeščajo, veliko možnih razcepov pa lahko vpliva na izbor.
Napovedno korist manjšega nabora preverimo s ponovnim učenjem in validacijo.

## 8. Samostojni poskusi
V spodnji funkciji so pripravljene tri družine modelov. Vsak poskus ponovno nauči kodiranje in model samo na učnih podatkih.
GBT uporablja isto pravilo zgodnjega ustavljanja. Rezultat z novim imenom se doda tabeli; isto ime prepiše prejšnji poskus.

In [ ]:
def nauci_poskus(ime,druzina,nastavitve=None,nabor=None):
    nabor=list(OSNOVNE if nabor is None else nabor)
    if not nabor or len(set(nabor))!=len(nabor) or not set(nabor)<=set(VSE):
        raise ValueError("Izberite veljaven, enoličen nabor vhodnih značilk.")
    nastavitve=nastavitve or {}
    prep,xt,xv,imena,izvor=pripravi(nabor)
    start=perf_counter()
    dodatno=dict(imena=imena,izvor=izvor)
    if druzina=="drevo":
        param={"max_depth":6,"min_samples_leaf":30,**nastavitve}
        m=DecisionTreeRegressor(criterion="squared_error",random_state=SEME,**param).fit(xt,y_train)
        cas=perf_counter()-start
        pt,pv=m.predict(xt),m.predict(xv)
    elif druzina=="gozd":
        param={**RF_PARAMETRI,**nastavitve}
        m=RandomForestRegressor(criterion="squared_error",random_state=SEME,n_jobs=N_JOBS,**param).fit(xt,y_train)
        cas=perf_counter()-start
        pt,pv=m.predict(xt),m.predict(xv)
    elif druzina=="gbt":
        param={**GBT_PARAMETRI,**nastavitve}
        if param.get("subsample",1)<1 and param.get("subsample_freq",0)==0:
            raise ValueError("Za vzorčenje vrstic nastavite tudi subsample_freq > 0.")
        m,k=nauci_gbt(xt,xv,param)
        cas=perf_counter()-start
        pt,pv=m.predict(xt,num_iteration=k),m.predict(xv,num_iteration=k)
        dodatno["iteracija"]=k
    else:
        raise ValueError("Družina mora biti drevo, gozd ali gbt.")
    return shrani(ime,m,prep,nabor,pt,pv,cas,**dodatno)

### Poskus A: nastavitve modela
**PREIZKUSITE:** spremenite družino in zanjo primerne nastavitve. Za začetek spreminjajte eno stvar.

| Družina | Nastavitve za preizkus |
| --- | --- |
| `drevo` | `max_depth`: 3, 6, 10; `min_samples_leaf`: 10, 50, 200 |
| `gozd` | `n_estimators`: 30, 80, 150; `max_features`: 0.5, 1.0; globina in velikost lista |
| `gbt` | `num_leaves`: 7, 15, 31; `learning_rate`: 0.02, 0.05, 0.15; `min_child_samples`: 30, 100, 300 |

Pri GBT manjša hitrost učenja lahko potrebuje več dreves. Primerjajte MAE, RMSE in čas.

In [ ]:
IME_POSKUSA="Moj GBT: manj listov"
DRUZINA="gbt"
MOJE_NASTAVITVE={"num_leaves":7}
moj=nauci_poskus(IME_POSKUSA,DRUZINA,MOJE_NASTAVITVE)
rezultati=primerjaj()

### Poskus B: manj ali več značilk
Za `top` izberemo značilke po skupnem gainu osnovnega GBT, za `vsi` pa celotno pripravljeno tabelo.
Pri `rocno` določite svoj seznam. Dodatne značilke lahko pomagajo, lahko pa povečajo prilagajanje in čas računanja.
Izbor na podlagi učnega modela še ne zagotavlja boljše validacijske napovedi.

In [ ]:
NACIN="top"  # "top", "vsi" ali "rocno"
TOP_K=6
ROCNI_NABOR=["vrednost","postnina","teza_g","zvezna_drzava"]
if NACIN=="top":
    if not 1<=TOP_K<=len(gbt_pom):raise ValueError("Izberite veljavno število značilk.")
    moj_nabor=gbt_pom["skupni gain"].nlargest(TOP_K).index.tolist()
elif NACIN=="vsi":moj_nabor=list(VSE)
elif NACIN=="rocno":moj_nabor=ROCNI_NABOR
else:raise ValueError("Izberite top, vsi ali rocno.")
print("Nabor:",moj_nabor)
izbor=nauci_poskus(f"GBT nabor {NACIN}: {len(moj_nabor)}","gbt",nabor=moj_nabor)
rezultati=primerjaj()

### Napovedovanje z izbranim postopkom
Pripravo in že naučeni model povežemo v Pipeline. Ni ponovnega učenja.
Vhod mora imeti isto obliko kot dopolnjena tabela iz prejšnjega poglavja.

In [ ]:
IZBRANI_MODEL="GBT"
izbrani=POSKUSI[IZBRANI_MODEL]
# Linearna/polinomska različica že vsebujeta svoj celotni postopek.
postopek=(izbrani["model"] if izbrani["priprava"] is None else
          Pipeline([("priprava",izbrani["priprava"]),("model",izbrani["model"])]))
dodatni={"num_iteration":izbrani["iteracija"]} if "iteracija" in izbrani else {}
novi=X_val[izbrani["nabor"]].iloc[:5]
napovedi=postopek.predict(novi,**dodatni)
np.testing.assert_allclose(napovedi,izbrani["p_val"][:5],atol=1e-8)
display(pd.DataFrame({"napoved dni":napovedi,"dejansko dni":y_val.iloc[:5].to_numpy()}).round(3))

## Zaključna primerjava vseh modelov na testu
Ko končate izbiro hiperparametrov in značilk na validaciji, nadaljujte v [zaključni primerjavi](../05_koncna_primerjava/A2_Koncna_primerjava_Olist.ipynb).
Tam vnesete svoje nastavitve in oznako udeleženca, naučite vse modele s skupnim semenom **42**, primerjate validacijske in testne rezultate ter izvozite oddajo za skupno primerjavo.
Test odprete šele v tem zaključnem koraku. Izvoženi rezultati vsebujejo tudi dejanske hiperparametre in uporabljena semena.